# FashionStore · Vestidor Virtual — Cuaderno de reparación y entrenamiento (Modal GPU)

**Proyecto:** SI-2 · Primer parcial · CU32 Vestidor Virtual
**Ejecutar en:** Modal → *Cuadernos* con GPU (A10G o L4). También corre en Colab sin cambios.

---

## 1. Diagnóstico del modelo anterior (`resultados_vton/`)

El modelo de `ia_vestidor/vton_transfer_learning_colab.py` recortaba la prenda nueva a la
silueta de la prenda que la persona ya llevaba puesta. **No era falta de entrenamiento.**
Eran cinco defectos de diseño:

| # | Dónde | Qué causaba |
|---|---|---|
| 1 | `mascara_anatomica = (ropa_sup \| brazos) & ~preservar` y `dice(wm, tm)` | Se le **ordenaba** al modelo que la prenda nueva tuviera la silueta de la vieja. Bug principal. |
| 2 | `area_borrar = dilatar(ropa_sup, 9)` | La agnóstica solo borra la ropa vieja; lo que cae fuera se repinta con la foto original. Mangas y vuelo imposibles. |
| 3 | `flujo = tanh(flujo) * 0.22` | Warp limitado a ±22% del cuadro: no puede expandir una prenda plana a un torso. |
| 4 | `l1_textura = |tryon − prenda_def| · mascara_ef` | Objetivo circular, sin verdad de terreno. Converge a *recolorear*, no a vestir. |
| 5 | `cuello ∈ preservar_fijo` | El cuello es zona preservada ⇒ un escote alto es **imposible de dibujar**. De ahí el fallo del cuello. |

También quedaron rotos en disco: `etapa2_v2/checkpoints/last.pth` (0 bytes) y
`onnx/vton_fashionstore.onnx` (3 KB de grafo + 184 KB de pesos, debería pesar ~150 MB → **export vacío**).

---

## 2. Qué hace este cuaderno

Arquitectura **híbrida**, porque los dos modos del vestidor tienen exigencias opuestas:

```
                      ┌─────────────────────────────────────────┐
  Foto subida  ─────► │ MODO FOTO   · difusión (CatVTON)        │ ~8-15 s  calidad alta
                      │   máscara agnóstica condicionada        │
                      └───────────────┬─────────────────────────┘
                                      │ destila 9×68 = 612 pares
                                      ▼
                      ┌─────────────────────────────────────────┐
  Cámara viva  ─────► │ MODO CÁMARA · red ligera ONNX           │ ~25-30 fps
                      │   entrenada imitando al modo foto       │
                      └─────────────────────────────────────────┘
```

**La destilación es la clave.** No tenías pares reales (9 cuerpos × 68 prendas, ninguna foto de
"esta persona con esta prenda puesta"), y sin pares no hay try-on posible. El modo foto los
**fabrica**: genera los 612 pares con tu catálogo real, y esos pares son la verdad de terreno
que le faltaba al modelo ligero. Por eso pediste que usáramos tus prendas y tus cuerpos — se
usan exactamente así.

**La corrección central** es `mascara_agnostica_por_prenda()` (celda 5): la zona que se borra
ya **no** es la silueta de la ropa vieja, sino una región derivada de la **pose** y ampliada
según el `neck_type` y el `sleeve_length` de la **prenda destino**, que ya tenías guardados en
`dataset_manifest.json` y no estabas usando.

**Castigos explícitos** (celda 11): el entrenamiento penaliza por nombre cada error que cometía
el modelo viejo — parecerse a la silueta anterior, equivocar el escote, recortar las mangas y
temblar entre fotogramas.

---
## Celda 1 · Entorno y GPU

En Modal: abre **Cuadernos → New notebook**, y en la rueda dentada elige GPU `A10G`
(suficiente y barata) y el volumen `fashionstore-vton` montado en `/vol`.

In [ ]:
import os, sys, subprocess, shutil, platform
from pathlib import Path

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=False)

EN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
EN_MODAL = Path("/vol").is_dir() or os.environ.get("MODAL_TASK_ID") is not None

print(f"Python   : {platform.python_version()}")
print(f"Entorno  : {'Modal' if EN_MODAL else 'Colab' if EN_COLAB else 'local'}")

try:
    import torch
    print(f"torch    : {torch.__version__}")
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU      : {p.name}  ·  {p.total_memory/1e9:.1f} GB")
    else:
        print("GPU      : NO DISPONIBLE  <-- detente y activa la GPU antes de seguir")
except ImportError:
    print("torch    : no instalado (la celda 2 lo instala)")

---
## Celda 2 · Dependencias

Se instalan una sola vez. En Modal quedan en la imagen del contenedor; si reinicias el
cuaderno vuelve a correr esta celda (es rápida, usa caché de pip).

In [ ]:
PAQUETES = [
    "torch --index-url https://download.pytorch.org/whl/cu121",
    "torchvision --index-url https://download.pytorch.org/whl/cu121",
    "diffusers==0.31.0",
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "huggingface_hub==0.25.2",
    "safetensors",
    "opencv-python-headless",
    "mediapipe==0.10.14",
    "onnx==1.16.2",
    "onnxruntime-gpu",
    "matplotlib",
    "tqdm",
    "scikit-image",
]

import importlib
YA = {"torch": "torch", "diffusers": "diffusers", "transformers": "transformers",
      "cv2": "opencv-python-headless", "mediapipe": "mediapipe", "onnx": "onnx"}
faltan = [m for m, _ in YA.items() if importlib.util.find_spec(m) is None]

if faltan:
    print(f"Faltan: {faltan} -> instalando todo")
    for p in PAQUETES:
        sh(f"{sys.executable} -m pip install -q {p}")
    print("\n>>> Reinicia el kernel del cuaderno y vuelve a correr desde la celda 1 <<<")
else:
    print("Dependencias completas.")

---
## Celda 3 · Datos: catálogo y cuerpos

### Subir los datos al volumen de Modal (una sola vez, **desde tu PC**, no aquí)

```powershell
pip install modal
python -m modal setup

modal volume create fashionstore-vton
modal volume put fashionstore-vton "C:\Users\MARILYN\Documents\Carpeta Esther\Semestre 2-2026\SI 2\Primer_parcial\entrenamiento_ia_ropa" /entrenamiento_ia_ropa
```

Son ~70 MB de imágenes, tarda un par de minutos. **No subas `resultados_vton/`**: pesa 2.5 GB
y no lo vamos a reutilizar (los checkpoints viejos arrastran el bug nº1).

Luego, en la configuración del cuaderno de Modal, monta `fashionstore-vton` en `/vol`.

In [ ]:
import json, csv, unicodedata, hashlib, random
from collections import Counter

# --- Localizar la raiz de datos -------------------------------------------------
CANDIDATOS = [
    Path("/vol"),                                   # Modal (volumen montado)
    Path("/content/drive/MyDrive/entrenamiento_ia_vestidor"),
    Path("/content/drive/MyDrive"),
    Path(".."), Path("."),
]
BASE = None
for c in CANDIDATOS:
    if (c / "entrenamiento_ia_ropa").is_dir():
        BASE = c
        break

if BASE is None and EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for c in CANDIDATOS:
        if (c / "entrenamiento_ia_ropa").is_dir():
            BASE = c
            break

assert BASE is not None, (
    "No encuentro 'entrenamiento_ia_ropa/'. En Modal: monta el volumen fashionstore-vton "
    "en /vol (ver instrucciones arriba)."
)

DIR_ROPA    = BASE / "entrenamiento_ia_ropa"
DIR_CUERPOS = DIR_ROPA / "cuerpos"
SALIDA      = BASE / "resultados_vton_v3"          # NUEVA carpeta: no tocamos la anterior
CACHE       = SALIDA / "cache"
DIR_PARES   = SALIDA / "pares_destilados"
for p in (SALIDA, CACHE, DIR_PARES):
    p.mkdir(parents=True, exist_ok=True)

print(f"BASE   : {BASE.resolve()}")
print(f"SALIDA : {SALIDA.resolve()}")

# --- Cargar catalogo con sus metadatos ------------------------------------------
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def norm(s):
    """Minusculas sin acentos, para comparar texto del manifiesto de forma robusta."""
    s = unicodedata.normalize("NFD", str(s or ""))
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.lower().strip()

manifest = json.loads((DIR_ROPA / "dataset_manifest.json").read_text(encoding="utf-8"))

# El manifiesto esta indexado por el nombre ORIGINAL; lo reindexamos por clean_name.
POR_LIMPIO = {}
for orig, meta in manifest.items():
    limpio = meta.get("clean_name") or orig
    POR_LIMPIO[limpio] = dict(meta, original=orig)

# Carpetas que NO son catalogo: resultados_vton guarda graficas .png que se colaban
# como si fueran prendas (se llegaba a destilar una figura de matplotlib).
NO_CATALOGO = {"cuerpos", "resultados_vton", "cache", "onnx", "hf"}

PRENDAS = []   # (Path, meta)
descartadas = []
for sub in sorted(DIR_ROPA.iterdir()):
    if (not sub.is_dir() or norm(sub.name) in NO_CATALOGO
            or norm(sub.name).startswith("resultados") or norm(sub.name).startswith(".")):
        continue
    for img in sorted(sub.iterdir()):
        if img.suffix.lower() not in EXT or img.name.endswith("_parse.png"):
            continue
        if img.name not in POR_LIMPIO:      # solo prendas del manifiesto oficial
            descartadas.append(img.name)
            continue
        meta = dict(POR_LIMPIO[img.name])
        meta.setdefault("category", sub.name)
        meta.setdefault("neck_type", "N/A")
        meta.setdefault("sleeve_length", "N/A")
        meta.setdefault("parent_category", "Ropa Superior")
        meta["archivo"] = img.name
        PRENDAS.append((img, meta))

# --- Cargar cuerpos --------------------------------------------------------------
CUERPOS = []
csv_cuerpos = DIR_CUERPOS / "cuerpos_metadata.csv"
if csv_cuerpos.exists():
    with open(csv_cuerpos, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            p = DIR_CUERPOS / row.get("filename", "")
            if p.exists():
                CUERPOS.append((p, row))
for img in sorted(DIR_CUERPOS.iterdir()):
    if img.suffix.lower() in EXT and not any(img == c[0] for c in CUERPOS):
        CUERPOS.append((img, {"name": img.stem, "body_type": "N/A"}))

print(f"\nCuerpos : {len(CUERPOS)}")
for p, m in CUERPOS:
    print(f"   - {m.get('name','?'):42s} {m.get('body_type','')}")

if descartadas:
    print("Descartadas (fuera del manifiesto):", descartadas)
cats = Counter(m["category"] for _, m in PRENDAS)
print(f"\nPrendas : {len(PRENDAS)}")
for c, n in sorted(cats.items()):
    print(f"   - {c:28s} {n}")
print(f"\nPares posibles a destilar: {len(CUERPOS)} x {len(PRENDAS)} = {len(CUERPOS)*len(PRENDAS)}")

---
## Celda 4 · Parser humano + pose

Dos modelos de análisis, ambos congelados (no se entrenan):

- **SegFormer B2 Clothes** — segmentación semántica en 18 clases (cara, pelo, ropa superior,
  brazos, piernas...). Es el mismo que ya usabas; ese no tenía ningún problema.
- **MediaPipe Pose** — 33 puntos del esqueleto. **Esto es lo nuevo.** El modelo anterior no
  usaba pose en ningún momento, y por eso la única referencia geométrica que tenía era la ropa
  que la persona ya llevaba. De ahí salía el bug nº1.

El resultado se cachea en el volumen: cada imagen se analiza una sola vez.

In [ ]:
import numpy as np, torch, cv2
from PIL import Image, ImageOps
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
W, H = 384, 512          # el doble que el modelo viejo (192x256): el cuello necesita resolucion

L = {"background":0,"hat":1,"hair":2,"sunglasses":3,"upper-clothes":4,"skirt":5,"pants":6,
     "dress":7,"belt":8,"left-shoe":9,"right-shoe":10,"face":11,"left-leg":12,"right-leg":13,
     "left-arm":14,"right-arm":15,"bag":16,"scarf":17}

_parser = {"proc": None, "net": None}

def parser():
    if _parser["net"] is None:
        mid = "mattmdjaga/segformer_b2_clothes"
        _parser["proc"] = SegformerImageProcessor.from_pretrained(mid)
        _parser["net"]  = SegformerForSemanticSegmentation.from_pretrained(mid).to(DEV).eval()
    return _parser["proc"], _parser["net"]

@torch.no_grad()
def segmentar(img_pil):
    proc, net = parser()
    ent = proc(images=img_pil, return_tensors="pt").to(DEV)
    logits = net(**ent).logits
    logits = torch.nn.functional.interpolate(
        logits, size=(img_pil.height, img_pil.width), mode="bilinear", align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)

# --- Pose -------------------------------------------------------------------------
import mediapipe as mp
_pose = None

def pose_de(img_pil):
    """33 landmarks en pixeles. Devuelve None si no detecta persona."""
    global _pose
    if _pose is None:
        _pose = mp.solutions.pose.Pose(static_image_mode=True, model_complexity=2,
                                       enable_segmentation=False, min_detection_confidence=0.4)
    arr = np.asarray(img_pil.convert("RGB"))
    r = _pose.process(arr)
    if not r.pose_landmarks:
        return None
    h, w = arr.shape[:2]
    return np.array([[lm.x * w, lm.y * h, lm.visibility] for lm in r.pose_landmarks.landmark],
                    dtype=np.float32)

# Indices MediaPipe que usamos
P = dict(nariz=0, boca_i=9, boca_d=10, hombro_i=11, hombro_d=12, codo_i=13, codo_d=14,
         muneca_i=15, muneca_d=16, cadera_i=23, cadera_d=24, rodilla_i=25, rodilla_d=26,
         tobillo_i=27, tobillo_d=28)

def letterbox(im, w=W, h=H, relleno=(255, 255, 255)):
    """Encaja la imagen en w x h sin deformarla, rellenando con blanco."""
    im = im.convert("RGB")
    e = min(w / im.width, h / im.height)
    nv = im.resize((max(1, int(im.width * e)), max(1, int(im.height * e))), Image.LANCZOS)
    lienzo = Image.new("RGB", (w, h), relleno)
    lienzo.paste(nv, ((w - nv.width) // 2, (h - nv.height) // 2))
    return lienzo

def clave(p):
    return hashlib.md5(str(Path(p).name).encode()).hexdigest()[:16]

def analizar(ruta, forzar=False):
    """Devuelve (img_pil, parse, pose) cacheado en el volumen."""
    k = clave(ruta)
    f_img, f_par, f_pos = CACHE / f"{k}.png", CACHE / f"{k}_parse.png", CACHE / f"{k}_pose.npy"
    if not forzar and f_img.exists() and f_par.exists():
        img = Image.open(f_img).convert("RGB")
        par = np.array(Image.open(f_par), dtype=np.uint8)
        pos = np.load(f_pos) if f_pos.exists() else None
        return img, par, pos
    img = letterbox(Image.open(ruta))
    par = segmentar(img)
    pos = pose_de(img)
    img.save(f_img)
    Image.fromarray(par).save(f_par)
    if pos is not None:
        np.save(f_pos, pos)
    return img, par, pos

from tqdm.auto import tqdm
print("Analizando cuerpos...")
for p, _ in tqdm(CUERPOS):
    analizar(p)
print("Analizando prendas...")
for p, _ in tqdm(PRENDAS):
    analizar(p)

sin_pose = [m.get("name") for (p, m) in CUERPOS if analizar(p)[2] is None]
if sin_pose:
    print(f"\nAVISO - sin pose detectada: {sin_pose}")
    print("Esos cuerpos usaran la mascara de respaldo (menos precisa).")
else:
    print(f"\nPose detectada en los {len(CUERPOS)} cuerpos.")

---
# Celda 5 · ⭐ La corrección: máscara agnóstica condicionada a la prenda destino

Esta celda es la reparación del modelo. Todo lo demás se apoya en ella.

### El principio

> La región que se borra del cuerpo debe ser un **superconjunto** de la silueta que la prenda
> nueva podría ocupar. Si se borra menos, la prenda se recorta. Nunca debe depender de la
> prenda que la persona ya lleva puesta.

El modelo anterior borraba `dilatar(ropa_vieja, 9)`. Si la persona llevaba un top de tirantes
y le pones una camiseta, los píxeles de las mangas **caen fuera de la zona borrada**, y en la
composición final se repintan con la foto original. Por eso salía un top de tirantes negro:
la camiseta sí se generaba, pero se recortaba contra el borde de la máscara.

### Cómo se calcula ahora

Tres fuentes, ninguna de ellas la ropa vieja:

1. **Pose** (MediaPipe) → hombros, codos, muñecas, caderas, barbilla. Da el esqueleto del torso
   y los brazos, independiente de la ropa.
2. **`neck_type` de la prenda destino** → hasta dónde se libera hacia la barbilla.
   `Cuello Alto con Cierre 1/4` libera hasta la barbilla; `Escote en V Profundo` casi nada.
3. **`sleeve_length` de la prenda destino** → qué fracción del brazo se libera.
   `Manga Larga` hasta la muñeca; `Sin Mangas` solo el hombro.

Los dos metadatos ya estaban en tu `dataset_manifest.json` desde el principio.

```
   barbilla ·······           cuello=1.00  (cuello alto / tortuga)
            ·······           cuello=0.85  (camisero / solapa / polo)
   hombros  ███████           cuello=0.70  (redondo / barco / cuadrado)
            ███████           cuello=0.35  (escote en V)
            ███████           cuello=0.15  (V profundo)
   codos    ██ manga=0.40 (corta / cap)
   muñecas  █  manga=1.00 (larga)
   caderas  ███████           largo=0.42  (ropa superior)
   rodillas ███████           largo=0.75  (vestido / enterizo)
```

Y por último se **une** con la ropa vieja dilatada: no para dar forma, sino para garantizar
que la prenda anterior desaparezca entera aunque sobresalga del recuadro geométrico.
Ese es el cambio de rol — de *plantilla* a *garantía de borrado*.

In [ ]:
# =====================================================================================
#  mascara_agnostica_por_prenda()  --  reemplaza a derivar_mascaras_cuerpo() del script viejo
# =====================================================================================

def plan_de_borrado(meta):
    """Traduce los metadatos de la prenda a tres fracciones geometricas.

    cuello : 0=no subir del hombro, 1=liberar hasta la barbilla
    manga  : fraccion del brazo (hombro->muneca) que se libera
    largo  : fraccion del cuerpo (hombro->tobillo) que se libera hacia abajo
    """
    n, s = norm(meta.get("neck_type")), norm(meta.get("sleeve_length"))
    cat, padre = norm(meta.get("category")), norm(meta.get("parent_category"))

    # --- cuello / escote ---
    if any(k in n for k in ("alto", "tortuga", "cierre 1/4", "cerrado")):
        cuello = 1.00
    elif any(k in n for k in ("camisero", "solapa", "polo")):
        cuello = 0.85
    elif "henley" in n:            # henley = redondo con botonadura, no camisero
        cuello = 0.70
    elif "profundo" in n:
        cuello = 0.15
    elif "en v" in n or n.startswith("v "):
        cuello = 0.35
    elif any(k in n for k in ("corazon", "tirantes", "asimetrico", "un hombro", "peto")):
        cuello = 0.45
    elif any(k in n for k in ("redondo", "barco", "recto", "cuadrado")):
        cuello = 0.70
    else:
        cuello = 0.70                      # N/A -> valor seguro (amplio)

    # --- mangas ---
    if "larga" in s:
        manga = 1.00
    elif "sin mangas" in s and "cap" not in s:
        manga = 0.14
    elif any(k in s for k in ("corta", "cap", "abullonada", "campana", "mariposa", "volantes")):
        manga = 0.45
    else:
        manga = 0.45

    # Chaquetas y cardigans van encima de todo: liberan mas que su escote nominal.
    if "chaqueta" in cat or "exterior" in padre or "cardigan" in norm(meta.get("archivo", "")):
        cuello, manga = max(cuello, 0.80), max(manga, 0.85)

    # --- largo ---
    if "una pieza" in padre or any(k in cat for k in ("vestido", "enterizo", "mono")):
        largo = 0.78                        # vestidos y enterizos bajan hasta la rodilla
    elif "inferior" in padre or any(k in cat for k in ("jean", "pantalon", "falda")):
        largo = 1.00                        # cubre piernas enteras
    elif "descanso" in padre or "pijama" in cat:
        largo = 0.78
    else:
        largo = 0.42                        # ropa superior: hasta un palmo bajo la cadera

    solo_inferior = ("inferior" in padre) or any(k in cat for k in ("jean", "pantalon", "falda"))
    return dict(cuello=cuello, manga=manga, largo=largo, solo_inferior=solo_inferior)


def _capsula(lienzo, a, b, radio):
    """Pinta un segmento grueso con extremos redondeados (un brazo)."""
    a, b = tuple(np.round(a[:2]).astype(int)), tuple(np.round(b[:2]).astype(int))
    cv2.line(lienzo, a, b, 1, thickness=int(max(2, radio * 2)), lineType=cv2.LINE_AA)
    cv2.circle(lienzo, a, int(radio), 1, -1, lineType=cv2.LINE_AA)
    cv2.circle(lienzo, b, int(radio), 1, -1, lineType=cv2.LINE_AA)


def mascara_agnostica_por_prenda(parse, pose, meta, margen=1.0):
    """Zona a regenerar. Depende de la POSE y de la PRENDA DESTINO, nunca de la ropa actual.

    Devuelve (mascara uint8 {0,1}, plan).
    """
    h, w = parse.shape
    plan = plan_de_borrado(meta)
    zona = np.zeros((h, w), np.uint8)

    cara_pelo = np.isin(parse, [L["hat"], L["hair"], L["sunglasses"], L["face"]])
    ropa_sup  = np.isin(parse, [L["upper-clothes"], L["dress"]])
    ropa_inf  = np.isin(parse, [L["skirt"], L["pants"], L["belt"]])

    if pose is not None:
        pt = lambda k: pose[P[k]][:2]
        hi, hd = pt("hombro_i"), pt("hombro_d")
        ci, cd = pt("cadera_i"), pt("cadera_d")
        ancho = float(np.linalg.norm(hi - hd)) or (w * 0.25)

        # Barbilla: bajo la boca, o estimada bajo la nariz si no hay boca visible.
        boca = (pt("boca_i") + pt("boca_d")) / 2.0
        barbilla = boca + np.array([0.0, 0.10 * ancho])

        y_hombro = (hi[1] + hd[1]) / 2.0
        y_tope = y_hombro + plan["cuello"] * (barbilla[1] - y_hombro)   # sube hacia la barbilla

        # Fondo del recuadro: interpolando hombro -> tobillo
        tob = (pt("tobillo_i") + pt("tobillo_d")) / 2.0
        y_fondo = y_hombro + plan["largo"] * (tob[1] - y_hombro)
        y_fondo = min(y_fondo, h - 1.0)

        ex_h = 0.20 * ancho * margen        # ensanchar hombros (prendas oversize)
        ex_c = 0.18 * ancho * margen        # ensanchar caderas (vuelo, peplum)
        izq, der = (hi, hd) if hi[0] < hd[0] else (hd, hi)
        c_izq, c_der = (ci, cd) if ci[0] < cd[0] else (cd, ci)

        torso = np.array([
            [izq[0] - ex_h,   y_tope],
            [der[0] + ex_h,   y_tope],
            [der[0] + ex_h,   (der[1] + c_der[1]) / 2.0],
            [c_der[0] + ex_c, y_fondo],
            [c_izq[0] - ex_c, y_fondo],
            [izq[0] - ex_h,   (izq[1] + c_izq[1]) / 2.0],
        ], np.int32)
        cv2.fillPoly(zona, [torso], 1, lineType=cv2.LINE_AA)

        # Brazos: capsula hombro->codo->muneca recortada a la fraccion 'manga'
        if not plan["solo_inferior"]:
            for lado in ("i", "d"):
                ho, co, mu = pt(f"hombro_{lado}"), pt(f"codo_{lado}"), pt(f"muneca_{lado}")
                l1, l2 = np.linalg.norm(co - ho), np.linalg.norm(mu - co)
                total = l1 + l2 + 1e-6
                alcance = plan["manga"] * total
                radio = 0.17 * ancho * margen
                if alcance <= l1:
                    fin = ho + (co - ho) * (alcance / (l1 + 1e-6))
                    _capsula(zona, ho, fin, radio)
                else:
                    _capsula(zona, ho, co, radio)
                    fin = co + (mu - co) * ((alcance - l1) / (l2 + 1e-6))
                    _capsula(zona, co, fin, radio * 0.80)
    else:
        # Respaldo sin pose: recuadro sobre la ropa detectada, generosamente ampliado.
        ys, xs = np.nonzero(ropa_sup)
        if ys.size:
            y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
            alto_r, ancho_r = y1 - y0, x1 - x0
            y0 = max(0, int(y0 - plan["cuello"] * 0.30 * alto_r))
            y1 = min(h - 1, int(y0 + plan["largo"] / 0.42 * alto_r))
            x0 = max(0, int(x0 - 0.35 * ancho_r))
            x1 = min(w - 1, int(x1 + 0.35 * ancho_r))
            zona[y0:y1 + 1, x0:x1 + 1] = 1

    # GARANTIA: la ropa anterior desaparece entera aunque sobresalga del recuadro.
    # (union, no plantilla: aqui esta la diferencia con el modelo viejo)
    previa = ropa_sup if not plan["solo_inferior"] else ropa_inf
    previa = cv2.dilate(previa.astype(np.uint8), np.ones((11, 11), np.uint8), 1)
    zona = np.clip(zona + previa, 0, 1).astype(np.uint8)

    # Nunca se borra la cabeza.
    zona[cv2.dilate(cara_pelo.astype(np.uint8), np.ones((3, 3), np.uint8), 1) > 0] = 0

    # Ropa inferior intacta salvo que la prenda la sustituya.
    if plan["largo"] < 0.60 and not plan["solo_inferior"]:
        conservar = np.isin(parse, [L["skirt"], L["pants"], L["left-leg"], L["right-leg"],
                                    L["left-shoe"], L["right-shoe"]])
        zona[conservar] = 0

    zona = cv2.morphologyEx(zona, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    return zona, plan


GRIS = 128

def construir_agnostica(img_pil, zona):
    """Persona con la zona a regenerar pintada de gris neutro."""
    arr = np.asarray(img_pil).copy()
    arr[zona > 0] = GRIS
    return Image.fromarray(arr)


def mascara_prenda_plana(img_pil):
    """Silueta de la prenda de catalogo (fondo claro y uniforme -> umbral + mayor componente)."""
    arr = np.asarray(img_pil.convert("RGB")).astype(np.int16)
    borde = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    fondo = np.median(borde, axis=0)
    dist = np.abs(arr - fondo).sum(2)
    m = (dist > 38).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, etiq, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n > 1:
        mayor = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        m = (etiq == mayor).astype(np.uint8)
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))


print("Plan de borrado por prenda (muestra):\n")
print(f"{'prenda':44s} {'escote':26s} {'manga':22s} cue  man  lar")
for ruta, meta in PRENDAS[:14]:
    pl = plan_de_borrado(meta)
    print(f"{meta['archivo'][:43]:44s} {str(meta['neck_type'])[:25]:26s} "
          f"{str(meta['sleeve_length'])[:21]:22s} "
          f"{pl['cuello']:.2f} {pl['manga']:.2f} {pl['largo']:.2f}")

---
## Celda 6 · Verificación visual del arreglo

Compara, sobre el mismo cuerpo, la máscara vieja contra la nueva para tres prendas de
escote y manga muy distintos. **Esto es la prueba de que el bug está corregido:**

- La máscara **vieja** es idéntica en las tres columnas — no depende de la prenda, solo del
  bodysuit que la modelo lleva puesta. Por eso las tres prendas salían con forma de bodysuit.
- La máscara **nueva** cambia en cada columna: sube al cuello en la de cuello alto, se extiende
  por los brazos en la de manga larga, se queda pegada al torso en la de tirantes.

Si las tres columnas de la fila nueva salen iguales, algo falló (revisa que haya pose).

In [ ]:
import matplotlib.pyplot as plt

def mascara_vieja(parse):
    """Reproduccion del area_borrar del script anterior, para comparar."""
    ropa = np.isin(parse, [L["upper-clothes"], L["dress"]]).astype(np.uint8)
    brazos = np.isin(parse, [L["left-arm"], L["right-arm"]]).astype(np.uint8)
    cara = np.isin(parse, [L["hat"], L["hair"], L["sunglasses"], L["face"]])
    area = cv2.dilate(ropa, np.ones((9, 9), np.uint8), 1)
    area |= (brazos & cv2.dilate(ropa, np.ones((16, 16), np.uint8), 1))
    area[cara] = 0
    return area

def elegir(fragmento):
    for r, m in PRENDAS:
        if fragmento in m["archivo"]:
            return r, m
    raise KeyError(fragmento)

# Tres prendas de contraste maximo, verificadas contra dataset_manifest.json:
#   cuello 1.00 / 0.70 / 0.35   ·   manga 1.00 / 0.45 / 0.14
MUESTRA = ["sueter_marinero_rayas_cuello_zipper",           # cuello alto 1/4 + manga larga
           "camiseta_basica_ajustada_azul_marino",          # redondo + manga corta
           "vestido_mini_tirantes_crochet_marron_terracota"]  # escote en V + sin mangas

ruta_cuerpo = CUERPOS[4][0]
img_c, parse_c, pose_c = analizar(ruta_cuerpo)
vieja = mascara_vieja(parse_c)

fig, ax = plt.subplots(4, len(MUESTRA), figsize=(4.0 * len(MUESTRA), 15))
for j, frag in enumerate(MUESTRA):
    rp, mp = elegir(frag)
    img_p, _, _ = analizar(rp)
    nueva, plan = mascara_agnostica_por_prenda(parse_c, pose_c, mp)

    ax[0, j].imshow(img_p)
    ax[0, j].set_title(f"{mp['neck_type'][:22]}\n{mp['sleeve_length'][:22]}", fontsize=9)

    ax[1, j].imshow(construir_agnostica(img_c, vieja))
    ax[1, j].set_title("VIEJA (identica siempre)", fontsize=9, color="#b00")

    ax[2, j].imshow(construir_agnostica(img_c, nueva))
    ax[2, j].set_title(f"NUEVA  cue={plan['cuello']:.2f} man={plan['manga']:.2f}",
                       fontsize=9, color="#070")

    dif = np.zeros((*nueva.shape, 3), np.uint8)
    dif[..., 0] = vieja * 200          # rojo  = solo la vieja
    dif[..., 1] = nueva * 200          # verde = lo que la nueva ANADE (antes se recortaba)
    ax[3, j].imshow(dif)
    ax[3, j].set_title("verde = espacio ganado", fontsize=9)

for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.savefig(SALIDA / "verificacion_mascara.png", dpi=110, bbox_inches="tight")
plt.show()

areas = []
for frag in MUESTRA:
    _, mp = elegir(frag)
    nueva, _ = mascara_agnostica_por_prenda(parse_c, pose_c, mp)
    areas.append(float(nueva.mean()))
print(f"\nArea liberada por prenda: {[f'{a:.3f}' for a in areas]}")
print(f"Area de la mascara vieja : {vieja.mean():.3f}  (constante para las 3)")
assert max(areas) - min(areas) > 0.01, \
    "FALLO: la mascara no varia con la prenda. Revisa que analizar() devuelva pose."
print("\nOK - la mascara depende de la prenda destino. Bug 1, 2 y 5 corregidos en origen.")

---
# Celda 7 · MODO FOTO — difusión (CatVTON)

Para la foto subida por la clienta prima la calidad, no la velocidad. Usamos **CatVTON**:
Stable Diffusion inpainting especializado en try-on, con ~800 M de parámetros entrenables.

**Por qué esto sí resuelve el escote y las mangas** y el modelo anterior no: CatVTON recibe la
máscara de inpainting como *entrada* y regenera **todo** lo que hay dentro de ella, sin ninguna
pérdida que lo ate a la silueta previa. Combinado con la máscara de la celda 5 —que ya se
adapta al `neck_type` y al `sleeve_length` de la prenda— el cuello deja de ser un problema
estructural.

Se descarga ~5 GB de pesos la primera vez; quedan cacheados en el volumen.

In [ ]:
import torch
os.environ.setdefault("HF_HOME", str(SALIDA / "hf"))
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

DIR_CATVTON = SALIDA / "CatVTON"
if not DIR_CATVTON.exists():
    sh(f"git clone --depth 1 https://github.com/Zheng-Chong/CatVTON {DIR_CATVTON}")
if str(DIR_CATVTON) not in sys.path:
    sys.path.insert(0, str(DIR_CATVTON))

from huggingface_hub import snapshot_download
REPO_CATVTON = snapshot_download(repo_id="zhengchong/CatVTON")
print("Pesos CatVTON en:", REPO_CATVTON)

from model.pipeline import CatVTONPipeline

pipe_foto = CatVTONPipeline(
    base_ckpt   = "booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt   = REPO_CATVTON,
    attn_ckpt_version = "mix",
    weight_dtype = torch.float16,
    device      = "cuda",
    skip_safety_check = True,
)
print("Modo foto listo.")

In [ ]:
# ---------------------------------------------------------------------------------
#  probar_prenda() : la funcion que expondra el backend para el MODO FOTO
# ---------------------------------------------------------------------------------
W_DIF, H_DIF = 384, 512     # resolucion nativa de CatVTON

def _a_pil(x):
    return x if isinstance(x, Image.Image) else Image.open(x).convert("RGB")

@torch.no_grad()
def probar_prenda(foto_persona, ruta_prenda, meta_prenda=None, pasos=40, escala=2.5, semilla=42):
    """Coloca una prenda del catalogo sobre la foto de una persona.

    foto_persona : ruta o PIL.Image (foto subida por la clienta)
    ruta_prenda  : ruta a la imagen plana del catalogo
    meta_prenda  : fila del dataset_manifest (neck_type / sleeve_length). Si falta, se busca.
    """
    if meta_prenda is None:
        meta_prenda = POR_LIMPIO.get(Path(ruta_prenda).name, {})
        meta_prenda = dict(meta_prenda, category=Path(ruta_prenda).parent.name)
        meta_prenda.setdefault("neck_type", "N/A")
        meta_prenda.setdefault("sleeve_length", "N/A")
        meta_prenda.setdefault("parent_category", "Ropa Superior")

    persona = letterbox(_a_pil(foto_persona), W_DIF, H_DIF)
    parse = segmentar(persona)
    pose  = pose_de(persona)

    zona, plan = mascara_agnostica_por_prenda(parse, pose, meta_prenda)
    mask_pil = Image.fromarray((zona * 255).astype(np.uint8)).convert("L")

    prenda = letterbox(_a_pil(ruta_prenda), W_DIF, H_DIF)

    g = torch.Generator(device="cuda").manual_seed(semilla)
    salida = pipe_foto(persona, prenda, mask_pil,
                       num_inference_steps=pasos, guidance_scale=escala, generator=g)
    resultado = salida[0] if isinstance(salida, (list, tuple)) else salida

    return dict(resultado=resultado, persona=persona, prenda=prenda,
                mascara=mask_pil, zona=zona, parse=parse, pose=pose, plan=plan,
                meta=meta_prenda)


# --- Demo: el caso exacto que fallaba ------------------------------------------------
# Cuerpo con bodysuit de tirantes + camiseta de manga: antes salia "camiseta con forma
# de bodysuit". Ahora la manga tiene sitio donde dibujarse.
demo_cuerpo = CUERPOS[4][0]
demo = [elegir(f) for f in MUESTRA]

fig, ax = plt.subplots(2, len(demo) + 1, figsize=(3.6 * (len(demo) + 1), 10))
ax[0, 0].imshow(letterbox(Image.open(demo_cuerpo), W_DIF, H_DIF)); ax[0, 0].set_title("original")
ax[1, 0].axis("off")
for j, (rp, mp) in enumerate(demo, start=1):
    r = probar_prenda(demo_cuerpo, rp, mp)
    ax[0, j].imshow(r["prenda"]); ax[0, j].set_title(mp["archivo"][:24], fontsize=8)
    ax[1, j].imshow(r["resultado"]); ax[1, j].set_title("resultado", fontsize=9)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.savefig(SALIDA / "demo_modo_foto.png", dpi=110, bbox_inches="tight")
plt.show()

---
# Celda 8 · Destilación: fabricar los 612 pares que nunca tuviste

Aquí se resuelve el defecto nº4, el de fondo.

El modelo anterior no tenía verdad de terreno: sus pérdidas comparaban la salida contra su
propia prenda deformada, dentro de su propia máscara predicha. Un objetivo circular que se
minimiza recoloreando, que es exactamente lo que hacía.

Ahora el modo foto **genera** esa verdad de terreno: cada uno de tus 9 cuerpos con cada una de
tus 68 prendas = **612 pares reales** `(agnóstica, prenda, máscara) → resultado`. No es un
dataset público genérico: es tu catálogo y tus modelos, que es justamente lo que pediste.

Con 8 aumentaciones por par (espejo, escala, brillo, desplazamiento) el modelo ligero ve
~4900 ejemplos supervisados. Suficiente para un modelo de 1,8 M de parámetros.

**Coste:** ~10 s por par en A10G ⇒ **~1 h 45 min** de GPU. En Modal A10G son ~$1,9.
Guarda progreso cada par: si se corta, vuelve a correr la celda y continúa donde iba.

> Si quieres abaratar, baja `PASOS_DESTILACION` a 30 y pon `LIMITE_PRENDAS = 40`
> (las 40 prendas de torso, que son las que el modo cámara realmente usa).

In [ ]:
import time
PASOS_DESTILACION = 40
LIMITE_PRENDAS    = None        # p.ej. 40 para abaratar
SOLO_TORSO        = True        # el modo camara solo viste el torso

def es_de_torso(meta):
    padre = norm(meta.get("parent_category"))
    return "inferior" not in padre

prendas_dest = [(r, m) for r, m in PRENDAS if (not SOLO_TORSO or es_de_torso(m))]
if LIMITE_PRENDAS:
    prendas_dest = prendas_dest[:LIMITE_PRENDAS]

tareas = [(rc, mc, rp, mp) for rc, mc in CUERPOS for rp, mp in prendas_dest]
print(f"Pares a destilar: {len(tareas)}  ({len(CUERPOS)} cuerpos x {len(prendas_dest)} prendas)")

DIR_PARES.mkdir(parents=True, exist_ok=True)
hechos = {p.stem for p in DIR_PARES.glob("*.png") if p.stem.endswith("_res")}
print(f"Ya destilados: {len(hechos)}")

t0 = time.time()
for i, (rc, mc, rp, mp) in enumerate(tqdm(tareas)):
    nombre = f"{clave(rc)}__{clave(rp)}"
    if f"{nombre}_res" in hechos:
        continue
    try:
        r = probar_prenda(rc, rp, mp, pasos=PASOS_DESTILACION)
    except Exception as e:
        print(f"  fallo en {mp['archivo']}: {e}")
        continue

    # Se guarda TODO lo que el modelo ligero necesitara como entrada y como objetivo.
    r["resultado"].resize((W, H), Image.LANCZOS).save(DIR_PARES / f"{nombre}_res.png")
    construir_agnostica(r["persona"], r["zona"]).resize((W, H), Image.LANCZOS)\
        .save(DIR_PARES / f"{nombre}_agn.png")
    Image.fromarray((r["zona"] * 255).astype(np.uint8)).resize((W, H), Image.NEAREST)\
        .save(DIR_PARES / f"{nombre}_zona.png")
    r["persona"].resize((W, H), Image.LANCZOS).save(DIR_PARES / f"{nombre}_ori.png")
    Image.fromarray(r["parse"]).resize((W, H), Image.NEAREST)\
        .save(DIR_PARES / f"{nombre}_parse.png")
    if r["pose"] is not None:
        np.save(DIR_PARES / f"{nombre}_pose.npy", r["pose"])
    (DIR_PARES / f"{nombre}_meta.json").write_text(
        json.dumps({"cuerpo": rc.name, "prenda": rp.name, "plan": r["plan"],
                    "neck_type": mp.get("neck_type"), "sleeve_length": mp.get("sleeve_length")},
                   ensure_ascii=False), encoding="utf-8")

    if (i + 1) % 25 == 0:
        el = time.time() - t0
        print(f"  {i+1}/{len(tareas)}  ·  {el/60:.1f} min  ·  quedan ~{el/(i+1)*(len(tareas)-i-1)/60:.0f} min")

n_ok = len(list(DIR_PARES.glob("*_res.png")))
print(f"\nPares destilados en disco: {n_ok}")
assert n_ok >= 100, "Muy pocos pares; el entrenamiento del modo camara no dara resultado."

---
## Celda 9 · Dataset del modo cámara

Cada muestra que ve el modelo ligero:

| tensor | canales | de dónde sale |
|---|---|---|
| `agnostica` | 3 | persona con la zona en gris (celda 5) |
| `zona` | 1 | máscara a regenerar |
| `pose_mapa` | 3 | campos de distancia a torso, brazos y cuello — le da la geometría del cuerpo sin depender de la ropa |
| `prenda` | 3 | imagen plana del catálogo |
| `mascara_prenda` | 1 | silueta de la prenda |
| **`objetivo`** | 3 | **lo que produjo el modo foto** ← la verdad de terreno |
| `ropa_previa` | 1 | silueta de la ropa vieja, solo para **castigarla** (celda 11) |

Fíjate en la última fila: la silueta de la ropa anterior sigue entrando al entrenamiento,
pero ha cambiado de bando. Antes era el objetivo del Dice; ahora es lo que se penaliza.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn, torch.nn.functional as F

def mapa_de_pose(pose, h=H, w=W, esc_x=1.0, esc_y=1.0):
    """3 canales: torso, brazos, cuello. Da geometria corporal sin mirar la ropa."""
    m = np.zeros((3, h, w), np.float32)
    if pose is None:
        return m
    pt = lambda k: np.array([pose[P[k]][0] * esc_x, pose[P[k]][1] * esc_y])
    hi, hd = pt("hombro_i"), pt("hombro_d")
    ancho = float(np.linalg.norm(hi - hd)) or (w * 0.25)
    l_torso = np.zeros((h, w), np.uint8)
    cv2.fillPoly(l_torso, [np.array([hi, hd, pt("cadera_d"), pt("cadera_i")], np.int32)], 1)
    m[0] = cv2.GaussianBlur(l_torso.astype(np.float32), (0, 0), ancho * 0.12)
    l_br = np.zeros((h, w), np.uint8)
    for lado in ("i", "d"):
        for a, b in ((f"hombro_{lado}", f"codo_{lado}"), (f"codo_{lado}", f"muneca_{lado}")):
            cv2.line(l_br, tuple(pt(a).astype(int)), tuple(pt(b).astype(int)), 1,
                     int(max(2, ancho * 0.16)))
    m[1] = cv2.GaussianBlur(l_br.astype(np.float32), (0, 0), ancho * 0.10)
    l_cu = np.zeros((h, w), np.uint8)
    boca = (pt("boca_i") + pt("boca_d")) / 2.0
    cv2.line(l_cu, tuple(boca.astype(int)), tuple(((hi + hd) / 2).astype(int)), 1,
             int(max(2, ancho * 0.28)))
    m[2] = cv2.GaussianBlur(l_cu.astype(np.float32), (0, 0), ancho * 0.10)
    return np.clip(m, 0, 1)


def a_tensor(img):
    a = np.asarray(img.convert("RGB"), np.float32) / 127.5 - 1.0
    return torch.from_numpy(a).permute(2, 0, 1)

def a_imagen(t):
    a = ((t.detach().cpu().float().permute(1, 2, 0).numpy() + 1) * 127.5)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


class DatasetDestilado(Dataset):
    def __init__(self, nombres, entrenar=True):
        self.nombres, self.entrenar = nombres, entrenar

    def __len__(self):
        return len(self.nombres) * (8 if self.entrenar else 1)

    def __getitem__(self, i):
        nom = self.nombres[i % len(self.nombres)]
        d = DIR_PARES
        try:
            agn  = Image.open(d / f"{nom}_agn.png").convert("RGB")
            obj  = Image.open(d / f"{nom}_res.png").convert("RGB")
            zona = np.array(Image.open(d / f"{nom}_zona.png").convert("L")) > 127
            parse = np.array(Image.open(d / f"{nom}_parse.png"))
            meta = json.loads((d / f"{nom}_meta.json").read_text(encoding="utf-8"))
            f_pose = d / f"{nom}_pose.npy"
            pose = np.load(f_pose) if f_pose.exists() else None

            rp = next(r for r, m in PRENDAS if m["archivo"] == meta["prenda"])
            prenda_img = letterbox(Image.open(rp), W, H)
            mpr = mascara_prenda_plana(prenda_img)

            espejo = self.entrenar and random.random() < 0.5
            if espejo:
                agn, obj = ImageOps.mirror(agn), ImageOps.mirror(obj)
                prenda_img = ImageOps.mirror(prenda_img)
                zona = zona[:, ::-1].copy(); parse = parse[:, ::-1].copy()
                mpr = mpr[:, ::-1].copy()
                if pose is not None:
                    pose = pose.copy(); pose[:, 0] = W - pose[:, 0]

            if self.entrenar and random.random() < 0.5:
                f = 0.85 + random.random() * 0.3
                agn = Image.eval(agn, lambda v: min(255, int(v * f)))
                obj = Image.eval(obj, lambda v: min(255, int(v * f)))

            pm = mapa_de_pose(pose)
            # El escote y las mangas llevan peso extra: son los dos errores del modelo viejo.
            peso = np.ones((1, H, W), np.float32)
            peso += 5.0 * pm[2:3]                                   # banda del cuello
            brazos = np.isin(parse, [L["left-arm"], L["right-arm"]]).astype(np.float32)
            peso += 3.0 * (brazos * zona)[None]                     # mangas nuevas
            ropa_previa = np.isin(parse, [L["upper-clothes"], L["dress"]]).astype(np.float32)

            return dict(
                agnostica = a_tensor(agn),
                zona      = torch.from_numpy(zona.astype(np.float32))[None],
                pose_mapa = torch.from_numpy(pm),
                prenda    = a_tensor(prenda_img),
                mascara_prenda = torch.from_numpy(mpr.astype(np.float32))[None],
                objetivo  = a_tensor(obj),
                peso      = torch.from_numpy(peso),
                ropa_previa = torch.from_numpy(ropa_previa)[None],
            )
        except Exception:
            return None

def collate(b):
    b = [x for x in b if x is not None]
    return torch.utils.data.default_collate(b) if b else None

nombres = sorted({p.stem[:-4] for p in DIR_PARES.glob("*_res.png")})
random.Random(42).shuffle(nombres)
corte = max(1, int(len(nombres) * 0.12))
tr, va = DatasetDestilado(nombres[corte:], True), DatasetDestilado(nombres[:corte], False)
LOAD_TR = DataLoader(tr, batch_size=6, shuffle=True, num_workers=2, collate_fn=collate, drop_last=True)
LOAD_VA = DataLoader(va, batch_size=6, shuffle=False, num_workers=2, collate_fn=collate)
print(f"Pares: {len(nombres)}  ->  entreno {len(tr)} muestras, validacion {len(va)}")

---
# Celda 10 · MODO CÁMARA — arquitectura ligera

Objetivo: **25-30 fps en el móvil**. Eso descarta la difusión (8-15 s por fotograma) y también
las dos ResNet34 del modelo anterior (~44 M de parámetros). Presupuesto: **~1,8 M**.

```
 entradas (11 canales)          ┌──────────┐
 agnostica 3 ─┐                 │ encoder  │   4 escalas, canales 24/48/96/160
 zona      1 ─┤                 │ separable│   convolución separable (estilo MobileNet)
 pose_mapa 3 ─┼──► concat ─────►│          │
 prenda    3 ─┤                 └────┬─────┘
 masc_pr   1 ─┘                      │
                        ┌────────────┼────────────┐
                        ▼            ▼            ▼
                  cabeza FLUJO   decoder U-Net   cabeza ALFA
                  ±0.60 (era     salida RGB      fusión
                  ±0.22, bug 3)
```

**Tres diferencias con el modelo roto:**

1. `flujo = tanh(f) * 0.60` en vez de `* 0.22`. El límite anterior hacía físicamente imposible
   estirar una prenda plana hasta un torso o alargar una manga.
2. La composición final se hace **dentro de `zona`**, no dentro de la máscara deformada. Ya no
   hay forma de que la silueta previa recorte el resultado.
3. La cabeza de alfa aprende la fusión; no hay `conservar_total` cableado a mano repintando la
   foto original encima.

In [ ]:
class ConvSep(nn.Module):
    """Convolucion separable en profundidad: el grueso del ahorro para llegar a 30 fps."""
    def __init__(self, ent, sal, paso=1):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(ent, ent, 3, paso, 1, groups=ent, bias=False),
            nn.BatchNorm2d(ent), nn.ReLU6(inplace=True),
            nn.Conv2d(ent, sal, 1, bias=False),
            nn.BatchNorm2d(sal), nn.ReLU6(inplace=True))
    def forward(self, x):
        return self.f(x)


def rejilla(h, w, dev):
    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=dev),
                          torch.linspace(-1, 1, w, device=dev), indexing="ij")
    return torch.stack([x, y], -1)[None]


def deformar(x, flujo, relleno="zeros"):
    b, _, h, w = x.shape
    base = rejilla(h, w, x.device).expand(b, -1, -1, -1)
    f = F.interpolate(flujo, size=(h, w), mode="bilinear", align_corners=False)
    return F.grid_sample(x, base + f.permute(0, 2, 3, 1), mode="bilinear",
                         padding_mode=relleno, align_corners=False)


CANALES = [24, 48, 96, 160]

class VestidorCamara(nn.Module):
    """Red ligera para el modo camara en vivo. ~1.8 M de parametros."""
    ENTRADAS = 3 + 1 + 3 + 3 + 1      # agnostica, zona, pose, prenda, mascara_prenda

    def __init__(self):
        super().__init__()
        c = CANALES
        self.tallo = nn.Sequential(nn.Conv2d(self.ENTRADAS, c[0], 3, 2, 1, bias=False),
                                   nn.BatchNorm2d(c[0]), nn.ReLU6(inplace=True))
        self.e1, self.e2, self.e3 = ConvSep(c[0], c[1], 2), ConvSep(c[1], c[2], 2), ConvSep(c[2], c[3], 2)

        # Cabeza de flujo: estima la deformacion de la prenda en el nivel mas profundo.
        self.cab_flujo = nn.Sequential(ConvSep(c[3], 64), nn.Conv2d(64, 2, 3, 1, 1))
        nn.init.zeros_(self.cab_flujo[-1].weight); nn.init.zeros_(self.cab_flujo[-1].bias)

        self.d3 = ConvSep(c[3] + c[2], c[2])
        self.d2 = ConvSep(c[2] + c[1], c[1])
        self.d1 = ConvSep(c[1] + c[0], c[0])
        # Fusion final: ve la prenda ya deformada a resolucion completa.
        self.salida = nn.Sequential(ConvSep(c[0] + 3 + 1 + 3, 32), nn.Conv2d(32, 4, 3, 1, 1))

    def forward(self, agnostica, zona, pose_mapa, prenda, mascara_prenda):
        x = torch.cat([agnostica, zona, pose_mapa, prenda, mascara_prenda], 1)
        f0 = self.tallo(x); f1 = self.e1(f0); f2 = self.e2(f1); f3 = self.e3(f2)

        # BUG 3 CORREGIDO: rango de deformacion 0.60 (antes 0.22, incapaz de expandir).
        flujo = torch.tanh(self.cab_flujo(f3)) * 0.60
        flujo_full = F.interpolate(flujo, size=agnostica.shape[-2:], mode="bilinear", align_corners=False)
        masc_def = deformar(mascara_prenda, flujo_full, "zeros")
        prenda_def = deformar(prenda, flujo_full, "border") * masc_def + (1 - masc_def)

        d = self.d3(torch.cat([F.interpolate(f3, f2.shape[-2:], mode="bilinear", align_corners=False), f2], 1))
        d = self.d2(torch.cat([F.interpolate(d, f1.shape[-2:], mode="bilinear", align_corners=False), f1], 1))
        d = self.d1(torch.cat([F.interpolate(d, f0.shape[-2:], mode="bilinear", align_corners=False), f0], 1))
        d = F.interpolate(d, size=agnostica.shape[-2:], mode="bilinear", align_corners=False)

        o = self.salida(torch.cat([d, prenda_def, masc_def, agnostica], 1))
        render, alfa = torch.tanh(o[:, :3]), torch.sigmoid(o[:, 3:4])

        # Fusion aprendida entre la prenda deformada y el render del decoder.
        vestido = alfa * prenda_def + (1 - alfa) * render
        # BUG 1/2 CORREGIDOS: se compone dentro de ZONA (geometria del cuerpo y de la prenda
        # destino), NO dentro de la silueta de la ropa anterior.
        final = zona * vestido + (1 - zona) * agnostica
        return dict(final=final, render=render, alfa=alfa, flujo=flujo_full,
                    prenda_def=prenda_def, masc_def=masc_def)


modelo = VestidorCamara().to(DEV)
n_par = sum(p.numel() for p in modelo.parameters())
print(f"Parametros: {n_par/1e6:.2f} M   (el modelo anterior tenia ~44 M)")

with torch.no_grad():
    d = next(iter(LOAD_VA))
    d = {k: v.to(DEV) for k, v in d.items()}
    o = modelo(d["agnostica"], d["zona"], d["pose_mapa"], d["prenda"], d["mascara_prenda"])
    print("Salida:", {k: tuple(v.shape) for k, v in o.items()})

---
# Celda 11 · Castigos

Pediste explícitamente que el modelo fuera castigado cuando se equivoque. Cada fallo concreto
del modelo anterior tiene aquí su penalización con nombre propio:

| castigo | qué error castiga | peso |
|---|---|---|
| `castigo_forma_vieja` | que la prenda generada se parezca a la silueta de la ropa que la persona ya llevaba — **el bug principal, ahora invertido** | 12,0 |
| `castigo_escote` | error en la banda del cuello, donde fallaba siempre | 6,0 |
| `castigo_recorte` | generar menos área de prenda que la referencia (mangas comidas) | 8,0 |
| `castigo_temblor` | flujo poco suave ⇒ parpadeo entre fotogramas en el modo cámara | 1,5 |
| `castigo_fuga` | pintar fuera de `zona`, sobre la cara o el fondo | 10,0 |

El primero merece una nota. Antes era `dice = 1 − Dice(máscara_generada, silueta_vieja)`, es
decir *recompensa* por parecerse a la ropa anterior. Ahora se mide cuánto se solapa la prenda
generada con la vieja **por encima** de lo que se solapa la referencia del modo foto, y ese
exceso se penaliza. Si la referencia legítimamente ocupa la zona del bodysuit, no hay castigo;
si el modelo se está *pegando* a la silueta vieja, sí lo hay.

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

class PerdidaVGG(nn.Module):
    """Perceptual ligera: solo hasta relu3_3, suficiente y barata."""
    def __init__(self):
        super().__init__()
        self.capas = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:16].eval()
        for p in self.capas.parameters():
            p.requires_grad = False
        self.register_buffer("mu",  torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("sd",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.hitos = {3: 1.0, 8: 0.8, 15: 0.6}

    def forward(self, x, y):
        x = ((x + 1) / 2 - self.mu) / self.sd
        y = ((y + 1) / 2 - self.mu) / self.sd
        total = 0.0
        for i, capa in enumerate(self.capas):
            x, y = capa(x), capa(y)
            if i in self.hitos:
                total = total + self.hitos[i] * F.l1_loss(x, y.detach())
        return total

VGG = PerdidaVGG().to(DEV)


def perdidas(o, b):
    final, obj, zona = o["final"], b["objetivo"], b["zona"]
    peso, previa = b["peso"], b["ropa_previa"]
    area_z = zona.sum(dim=[1, 2, 3]) + 1e-5

    # --- reconstruccion contra la verdad de terreno destilada -----------------------
    l1  = (((final - obj).abs() * peso * zona).sum(dim=[1, 2, 3]) /
           (peso * zona).sum(dim=[1, 2, 3]).clamp_min(1e-5)).mean()
    vgg = VGG(final * zona, obj * zona)

    # --- CASTIGO 1: parecerse a la silueta de la ropa anterior ----------------------
    # Mascara de "hay prenda" = donde la salida se aparta del gris de la agnostica.
    gen = (final - b["agnostica"]).abs().mean(1, keepdim=True).clamp(0, 1) * zona
    ref = (obj   - b["agnostica"]).abs().mean(1, keepdim=True).clamp(0, 1) * zona
    solape_gen = (gen * previa).sum(dim=[1, 2, 3]) / area_z
    solape_ref = (ref * previa).sum(dim=[1, 2, 3]) / area_z
    castigo_forma_vieja = 12.0 * F.relu(solape_gen - solape_ref - 0.02).mean()

    # --- CASTIGO 2: el escote (banda del cuello) ------------------------------------
    cuello = b["pose_mapa"][:, 2:3] * zona
    castigo_escote = 6.0 * (((final - obj).abs() * cuello).sum(dim=[1, 2, 3]) /
                            cuello.sum(dim=[1, 2, 3]).clamp_min(1e-5)).mean()

    # --- CASTIGO 3: recortar la prenda (mangas comidas) -----------------------------
    castigo_recorte = 8.0 * F.relu(
        (ref.sum(dim=[1, 2, 3]) - gen.sum(dim=[1, 2, 3])) / area_z - 0.03).mean()

    # --- CASTIGO 4: temblor entre fotogramas (modo camara) --------------------------
    f = o["flujo"]
    tv = (f[:, :, 1:] - f[:, :, :-1]).abs().mean() + (f[:, :, :, 1:] - f[:, :, :, :-1]).abs().mean()
    castigo_temblor = 1.5 * tv

    # --- CASTIGO 5: pintar fuera de la zona (cara, pelo, fondo) ---------------------
    fuera = 1.0 - zona
    castigo_fuga = 10.0 * (((final - b["agnostica"]).abs() * fuera).sum(dim=[1, 2, 3]) /
                           fuera.sum(dim=[1, 2, 3]).clamp_min(1e-5)).mean()

    total = (3.0 * l1 + 2.0 * vgg + castigo_forma_vieja + castigo_escote +
             castigo_recorte + castigo_temblor + castigo_fuga)

    return total, dict(total=float(total), l1=float(l1), vgg=float(vgg),
                       forma_vieja=float(castigo_forma_vieja), escote=float(castigo_escote),
                       recorte=float(castigo_recorte), temblor=float(castigo_temblor),
                       fuga=float(castigo_fuga))

print("Castigos definidos. Vigila en las curvas que 'forma_vieja' y 'recorte' bajen a ~0.")

---
## Celda 12 · Entrenamiento del modo cámara

~35 min en A10G. Guarda checkpoint cada época **verificando que el archivo no quede vacío**
(el `last.pth` de 0 bytes del entrenamiento anterior nació de un guardado que falló en silencio).

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

EPOCAS = 60
DIR_CKPT = SALIDA / "camara" / "checkpoints"
DIR_CKPT.mkdir(parents=True, exist_ok=True)

opt = torch.optim.AdamW(modelo.parameters(), lr=2e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=EPOCAS * max(1, len(LOAD_TR)),
                                            pct_start=0.15)
scaler = torch.cuda.amp.GradScaler(enabled=DEV.type == "cuda")
hist = defaultdict(list)
mejor = float("inf")

def guardar(ruta, estado):
    """Guarda y comprueba que el archivo tenga tamano real."""
    tmp = Path(str(ruta) + ".tmp")
    torch.save(estado, tmp)
    assert tmp.stat().st_size > 1_000_000, f"checkpoint vacio: {tmp.stat().st_size} bytes"
    tmp.replace(ruta)
    return ruta.stat().st_size

for ep in range(1, EPOCAS + 1):
    modelo.train(); acc, n = defaultdict(float), 0
    for lote in LOAD_TR:
        if lote is None: continue
        b = {k: v.to(DEV, non_blocking=True) for k, v in lote.items()}
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", torch.float16, enabled=DEV.type == "cuda"):
            o = modelo(b["agnostica"], b["zona"], b["pose_mapa"], b["prenda"], b["mascara_prenda"])
            loss, met = perdidas(o, b)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 3.0)
        scaler.step(opt); scaler.update(); sched.step()
        bs = b["agnostica"].shape[0]
        for k, v in met.items(): acc[k] += v * bs
        n += bs
    tr_m = {k: v / max(1, n) for k, v in acc.items()}

    modelo.eval(); acc, n, demo = defaultdict(float), 0, None
    with torch.no_grad():
        for lote in LOAD_VA:
            if lote is None: continue
            b = {k: v.to(DEV) for k, v in lote.items()}
            if demo is None: demo = b
            with torch.autocast("cuda", torch.float16, enabled=DEV.type == "cuda"):
                o = modelo(b["agnostica"], b["zona"], b["pose_mapa"], b["prenda"], b["mascara_prenda"])
                _, met = perdidas(o, b)
            bs = b["agnostica"].shape[0]
            for k, v in met.items(): acc[k] += v * bs
            n += bs
    va_m = {k: v / max(1, n) for k, v in acc.items()}

    for k in tr_m: hist[f"tr_{k}"].append(tr_m[k])
    for k in va_m: hist[f"va_{k}"].append(va_m[k])

    print(f"ep {ep:3d}/{EPOCAS}  tr {tr_m['total']:.4f}  va {va_m['total']:.4f}  |  "
          f"l1 {va_m['l1']:.3f}  forma_vieja {va_m['forma_vieja']:.4f}  "
          f"escote {va_m['escote']:.3f}  recorte {va_m['recorte']:.4f}")

    if va_m["total"] < mejor:
        mejor = va_m["total"]
        kb = guardar(DIR_CKPT / "mejor.pth",
                     dict(modelo=modelo.state_dict(), epoca=ep, loss=mejor,
                          W=W, H=H, canales=CANALES)) / 1e6
        print(f"        nuevo mejor -> mejor.pth ({kb:.1f} MB)")

    if ep % 10 == 0 or ep == EPOCAS:
        with torch.no_grad():
            o = modelo(demo["agnostica"], demo["zona"], demo["pose_mapa"],
                       demo["prenda"], demo["mascara_prenda"])
        k = min(3, demo["agnostica"].shape[0])
        fig, ax = plt.subplots(k, 4, figsize=(13, 4.2 * k))
        ax = ax.reshape(k, 4)
        for i in range(k):
            for j, (t, tit) in enumerate([(demo["agnostica"][i], "agnostica"),
                                          (demo["prenda"][i], "prenda"),
                                          (o["final"][i], "modo camara"),
                                          (demo["objetivo"][i], "objetivo (modo foto)")]):
                ax[i, j].imshow(a_imagen(t)); ax[i, j].set_title(tit, fontsize=9); ax[i, j].axis("off")
        plt.tight_layout()
        plt.savefig(SALIDA / "camara" / f"visual_epoca_{ep:03d}.png", dpi=95, bbox_inches="tight")
        plt.show()

# --- curvas ----------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].plot(hist["tr_total"], label="entreno"); ax[0].plot(hist["va_total"], label="validacion")
ax[0].set_title("perdida total"); ax[0].legend(); ax[0].grid(alpha=.3)
for k in ("va_l1", "va_vgg", "va_escote"):
    ax[1].plot(hist[k], label=k)
ax[1].set_title("reconstruccion y escote"); ax[1].legend(); ax[1].grid(alpha=.3)
for k in ("va_forma_vieja", "va_recorte", "va_fuga"):
    ax[2].plot(hist[k], label=k)
ax[2].set_title("castigos - deben tender a 0"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig(SALIDA / "camara" / "curvas.png", dpi=110); plt.show()

print(f"\nMejor validacion: {mejor:.4f}")
print(f"forma_vieja final: {hist['va_forma_vieja'][-1]:.4f}  (si queda alto, el modelo aun "
      f"se pega a la silueta previa: sube el peso a 20.0 y reentrena)")

---
# Celda 13 · Export ONNX **verificado**

El export anterior produjo `vton_fashionstore.onnx` de 3 KB con 184 KB de pesos, cuando debía
rondar los 150 MB: **el grafo salió sin pesos** y lo que corría en producción no era el modelo
entrenado. Esta celda no confía en que el export haya ido bien: lo comprueba.

Tres verificaciones, todas con `assert`:

1. **Tamaño** — los bytes en disco coinciden con los parámetros del modelo (±25 %).
2. **Numérica** — la salida de ONNX Runtime coincide con la de PyTorch (error máx < 0,02).
3. **Velocidad** — se mide el fps real. Si baja de 20, el modo cámara no es viable en móvil.

In [ ]:
import onnx, onnxruntime as ort

DIR_ONNX = SALIDA / "onnx"
DIR_ONNX.mkdir(parents=True, exist_ok=True)
RUTA_ONNX = DIR_ONNX / "vestidor_camara.onnx"

ck = torch.load(DIR_CKPT / "mejor.pth", map_location="cpu")
modelo_exp = VestidorCamara()
modelo_exp.load_state_dict(ck["modelo"])
modelo_exp.eval()

class SoloFinal(nn.Module):
    """En produccion solo hace falta la imagen final."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, agnostica, zona, pose_mapa, prenda, mascara_prenda):
        return self.m(agnostica, zona, pose_mapa, prenda, mascara_prenda)["final"]

envoltorio = SoloFinal(modelo_exp).eval()
ej = (torch.randn(1, 3, H, W), torch.rand(1, 1, H, W), torch.rand(1, 3, H, W),
      torch.randn(1, 3, H, W), torch.rand(1, 1, H, W))

torch.onnx.export(
    envoltorio, ej, str(RUTA_ONNX),
    input_names=["agnostica", "zona", "pose_mapa", "prenda", "mascara_prenda"],
    output_names=["final"], opset_version=17, do_constant_folding=True,
    dynamic_axes={k: {0: "lote"} for k in
                  ["agnostica", "zona", "pose_mapa", "prenda", "mascara_prenda", "final"]},
)

# --- 1. Tamano --------------------------------------------------------------------
bytes_disco = sum(f.stat().st_size for f in DIR_ONNX.glob("vestidor_camara.onnx*"))
n_par = sum(p.numel() for p in modelo_exp.parameters())
esperado = n_par * 4
print(f"Parametros : {n_par:,}  ->  esperado ~{esperado/1e6:.1f} MB")
print(f"En disco   : {bytes_disco/1e6:.1f} MB")
assert bytes_disco > esperado * 0.75, (
    f"EXPORT VACIO: {bytes_disco/1e6:.2f} MB frente a {esperado/1e6:.1f} MB esperados. "
    "Es el mismo fallo del modelo anterior; no despliegues este archivo.")
onnx.checker.check_model(onnx.load(str(RUTA_ONNX)))
print("1/3 tamano y estructura OK")

# --- 2. Equivalencia numerica ------------------------------------------------------
prov = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
ses = ort.InferenceSession(str(RUTA_ONNX), providers=prov)
nombres_ent = [e.name for e in ses.get_inputs()]
ent_np = {n: t.numpy() for n, t in zip(nombres_ent, ej)}
sal_onnx = ses.run(None, ent_np)[0]
with torch.no_grad():
    sal_torch = envoltorio(*ej).numpy()
err = float(np.abs(sal_onnx - sal_torch).max())
print(f"2/3 error maximo PyTorch vs ONNX: {err:.5f}")
assert err < 0.02, "El ONNX no reproduce al modelo; revisa el opset."

# --- 3. Velocidad ------------------------------------------------------------------
import time
for _ in range(5):
    ses.run(None, ent_np)
t0 = time.time()
N = 40
for _ in range(N):
    ses.run(None, ent_np)
fps = N / (time.time() - t0)
print(f"3/3 velocidad: {fps:.1f} fps @ {W}x{H}")
if fps < 20:
    print("    AVISO: por debajo de 20 fps. Para el movil baja a 256x192 o reduce CANALES.")

(DIR_ONNX / "vestidor_metadatos.json").write_text(json.dumps({
    "modelo": "vestidor_camara.onnx",
    "version": time.strftime("%Y%m%d-%H%M"),
    "arquitectura": "VestidorCamara - encoder separable + cabeza de flujo + fusion alfa",
    "parametros": int(n_par),
    "entrenado_por": "destilacion de CatVTON sobre el catalogo FashionStore",
    "pares_destilados": len(list(DIR_PARES.glob("*_res.png"))),
    "alto": H, "ancho": W,
    "entradas": {
        "agnostica": f"float32 [b,3,{H},{W}] en [-1,1] - persona con la zona en gris 128",
        "zona": f"float32 [b,1,{H},{W}] en {{0,1}} - mascara_agnostica_por_prenda()",
        "pose_mapa": f"float32 [b,3,{H},{W}] en [0,1] - torso, brazos, cuello",
        "prenda": f"float32 [b,3,{H},{W}] en [-1,1] - prenda plana sobre blanco",
        "mascara_prenda": f"float32 [b,1,{H},{W}] en {{0,1}}",
    },
    "salidas": {"final": f"float32 [b,3,{H},{W}] en [-1,1]"},
    "fps_medido": round(fps, 1),
    "verificado": {"bytes": bytes_disco, "error_max_vs_torch": err},
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nListo: {RUTA_ONNX}")

---
# Celda 14 · Servicio en Modal para el backend FastAPI

Escribe y despliega `app_vton.py`: dos endpoints web que el backend de FashionStore consume.

| endpoint | modo | modelo | latencia |
|---|---|---|---|
| `POST /probar-foto` | foto subida | CatVTON | ~8-15 s |
| `POST /probar-camara` | cámara en vivo | ONNX ligero | ~40 ms |

`scaledown_window=300` mantiene el contenedor vivo 5 min tras la última petición: la clienta
prueba varias prendas seguidas y solo la primera paga el arranque en frío.

**Ojo con el coste:** `min_containers=0` (por defecto) significa que no pagas nada mientras
nadie use el vestidor. No lo subas a 1 salvo el día de la defensa del parcial.

In [ ]:
# ---------------------------------------------------------------------------------
#  vton_comun.py : se genera A PARTIR DE ESTE CUADERNO con inspect.getsource, para
#  que el servicio use exactamente las mismas funciones con las que se entreno.
#  Si cambias la mascara aqui arriba, se propaga sola al despliegue.
# ---------------------------------------------------------------------------------
import inspect

COMPARTIDAS = [plan_de_borrado, _capsula, mascara_agnostica_por_prenda,
               construir_agnostica, mascara_prenda_plana, mapa_de_pose,
               letterbox, a_tensor, a_imagen, norm]

CABECERA = f"""# vton_comun.py - GENERADO por vestidor_virtual_modal.ipynb. No editar a mano.
import unicodedata, numpy as np, cv2, torch
from PIL import Image

W, H = {W}, {H}
GRIS = {GRIS}
L = {L!r}
P = {P!r}
"""

CODIGO_COMUN = CABECERA + "\n\n" + "\n\n".join(inspect.getsource(f) for f in COMPARTIDAS)
(SALIDA / "vton_comun.py").write_text(CODIGO_COMUN, encoding="utf-8")

import ast as _ast
_ast.parse(CODIGO_COMUN)      # falla aqui antes que en produccion
print(f"vton_comun.py generado: {len(COMPARTIDAS)} funciones, "
      f"{len(CODIGO_COMUN.splitlines())} lineas")


CODIGO_APP = r"""
# app_vton.py - Servicio del Vestidor Virtual FashionStore (CU32) en Modal
import io, json, base64
from pathlib import Path
import modal

app = modal.App("fashionstore-vestidor")
vol = modal.Volume.from_name("fashionstore-vton", create_if_missing=True)

imagen = (
    modal.Image.debian_slim(python_version="3.11")
    .apt_install("git", "libgl1", "libglib2.0-0")
    .pip_install(
        "torch", "torchvision", "diffusers==0.31.0", "transformers==4.44.2",
        "accelerate==0.34.2", "huggingface_hub==0.25.2", "safetensors",
        "opencv-python-headless", "mediapipe==0.10.14", "onnxruntime-gpu",
        "numpy", "pillow", "fastapi[standard]",
    )
    .env({"HF_HOME": "/vol/resultados_vton_v3/hf"})
)

RAIZ = Path("/vol")
V3 = RAIZ / "resultados_vton_v3"


@app.cls(image=imagen, gpu="A10G", volumes={"/vol": vol},
         scaledown_window=300, timeout=600)
class Vestidor:
    @modal.enter()
    def cargar(self):
        import sys, torch, onnxruntime as ort
        from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
        import mediapipe as mp

        self.dev = torch.device("cuda")
        mid = "mattmdjaga/segformer_b2_clothes"
        self.proc = SegformerImageProcessor.from_pretrained(mid)
        self.seg = SegformerForSemanticSegmentation.from_pretrained(mid).to(self.dev).eval()
        self.pose = mp.solutions.pose.Pose(static_image_mode=False, model_complexity=1,
                                           min_detection_confidence=0.4)

        sys.path.insert(0, str(V3))              # vton_comun.py (generado por el cuaderno)
        sys.path.insert(0, str(V3 / "CatVTON"))
        from model.pipeline import CatVTONPipeline
        from huggingface_hub import snapshot_download
        self.pipe = CatVTONPipeline(
            base_ckpt="booksforcharlie/stable-diffusion-inpainting",
            attn_ckpt=snapshot_download("zhengchong/CatVTON"),
            attn_ckpt_version="mix", weight_dtype=torch.float16,
            device="cuda", skip_safety_check=True)

        self.ort = ort.InferenceSession(
            str(V3 / "onnx" / "vestidor_camara.onnx"),
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

        self.manifest = json.loads(
            (RAIZ / "entrenamiento_ia_ropa" / "dataset_manifest.json").read_text(encoding="utf-8"))
        self.por_limpio = {v.get("clean_name", k): v for k, v in self.manifest.items()}
        print("Vestidor cargado.")

    # --- analisis comun a los dos modos -------------------------------------------
    def _analizar(self, img_bytes, nombre_prenda):
        # Devuelve (persona, prenda, zona, pose) usando las MISMAS funciones del cuaderno.
        import numpy as np, torch
        from PIL import Image
        import vton_comun as C

        persona = C.letterbox(Image.open(io.BytesIO(img_bytes)).convert("RGB"))

        meta = dict(self.por_limpio.get(nombre_prenda, {}))
        for k, v in (("neck_type", "N/A"), ("sleeve_length", "N/A"),
                     ("parent_category", "Ropa Superior"), ("category", "")):
            meta.setdefault(k, v)

        ent = self.proc(images=persona, return_tensors="pt").to(self.dev)
        with torch.no_grad():
            lg = self.seg(**ent).logits
        lg = torch.nn.functional.interpolate(lg, size=(persona.height, persona.width),
                                             mode="bilinear", align_corners=False)
        parse = lg.argmax(1)[0].cpu().numpy().astype(np.uint8)

        r = self.pose.process(np.asarray(persona))
        pose = None
        if r.pose_landmarks:
            h, w = persona.height, persona.width
            pose = np.array([[l.x * w, l.y * h, l.visibility]
                             for l in r.pose_landmarks.landmark], dtype=np.float32)

        zona, _ = C.mascara_agnostica_por_prenda(parse, pose, meta)

        ruta_p = next(RAIZ.joinpath("entrenamiento_ia_ropa").rglob(nombre_prenda))
        prenda = C.letterbox(Image.open(ruta_p))
        return persona, prenda, zona, pose, parse

    @modal.fastapi_endpoint(method="POST", docs=True)
    def probar_foto(self, datos: dict):
        # MODO FOTO: {foto_b64, prenda} -> {imagen_b64}. Difusion, ~8-15 s.
        from PIL import Image
        import numpy as np, torch
        persona, prenda, zona, _, _ = self._analizar(
            base64.b64decode(datos["foto_b64"]), datos["prenda"])
        mask = Image.fromarray((zona * 255).astype(np.uint8)).convert("L")
        g = torch.Generator(device="cuda").manual_seed(int(datos.get("semilla", 42)))
        salida = self.pipe(persona, prenda, mask, num_inference_steps=int(datos.get("pasos", 40)),
                           guidance_scale=2.5, generator=g)
        img = salida[0] if isinstance(salida, (list, tuple)) else salida
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=92)
        return {"imagen_b64": base64.b64encode(buf.getvalue()).decode(), "modo": "foto"}

    @modal.fastapi_endpoint(method="POST", docs=True)
    def probar_camara(self, datos: dict):
        # MODO CAMARA: {frame_b64, prenda} -> {imagen_b64}. Red ligera ONNX, ~40 ms.
        import numpy as np
        import vton_comun as C
        persona, prenda, zona, pose, _ = self._analizar(
            base64.b64decode(datos["frame_b64"]), datos["prenda"])

        agn = C.construir_agnostica(persona, zona)
        ent = {
            "agnostica":      C.a_tensor(agn).numpy()[None],
            "zona":           zona.astype(np.float32)[None, None],
            "pose_mapa":      C.mapa_de_pose(pose)[None],
            "prenda":         C.a_tensor(prenda).numpy()[None],
            "mascara_prenda": C.mascara_prenda_plana(prenda).astype(np.float32)[None, None],
        }
        final = self.ort.run(None, ent)[0][0]
        import torch
        img = C.a_imagen(torch.from_numpy(final))
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=80)
        return {"imagen_b64": base64.b64encode(buf.getvalue()).decode(), "modo": "camara"}


@app.local_entrypoint()
def main():
    print("Despliega con:  modal deploy app_vton.py")
"""

ruta_app = SALIDA / "app_vton.py"
ruta_app.write_text(CODIGO_APP, encoding="utf-8")
_ast.parse(CODIGO_APP)        # valida la app antes de intentar desplegarla
print(f"Escrito y validado: {ruta_app}")

print("""
Para desplegar, DESDE TU PC (no desde el cuaderno):

  modal volume get fashionstore-vton /resultados_vton_v3/app_vton.py .
  modal deploy app_vton.py

vton_comun.py NO se baja: el servicio lo lee del volumen en /vol/resultados_vton_v3/,
asi que siempre corre la misma mascara con la que se entreno.

Modal imprime dos URLs. Copialas a backend/.env:

  VTON_URL_FOTO=https://<usuario>--fashionstore-vestidor-vestidor-probar-foto.modal.run
  VTON_URL_CAMARA=https://<usuario>--fashionstore-vestidor-vestidor-probar-camara.modal.run
""")

---
## Celda 15 · Conectar con FashionStore

### Backend (`backend/app/packages/vestidor_virtual/`)

```python
# service.py
import base64, httpx
from app.core.config import settings

async def probar_prenda_foto(foto: bytes, nombre_prenda: str) -> bytes:
    """CU32 modo foto. La clienta sube su foto y recibe el resultado."""
    async with httpx.AsyncClient(timeout=120.0) as cli:
        r = await cli.post(settings.VTON_URL_FOTO, json={
            "foto_b64": base64.b64encode(foto).decode(),
            "prenda": nombre_prenda,
        })
        r.raise_for_status()
        return base64.b64decode(r.json()["imagen_b64"])
```

`nombre_prenda` es el `clean_name` del manifiesto, que ya es el nombre de archivo que guardas
en `products` — no hace falta tabla nueva.

### Móvil (`mobile/lib/src/packages/vestidor_virtual/`)

- **Modo foto:** `image_picker` → POST → mostrar. Pon un `CircularProgressIndicator`: son 8-15 s
  y sin aviso la clienta cree que se colgó.
- **Modo cámara:** `camera` con `startImageStream`, y **manda 1 de cada 3 fotogramas** por
  WebSocket. A 30 fps el móvil genera más peticiones de las que la red aguanta; con 10 fps
  enviados y el resultado interpolado se ve fluido y cuesta un tercio.

### Advertencia de coste

Cada petición del modo foto son ~12 s de A10G ≈ **$0,004**. El modo cámara a 10 fps es
**$0,019 por minuto** de uso continuo. Con $1 de crédito tienes ~50 min de cámara. Para la
demo del parcial va sobrado; para producción, pon un límite por sesión en el backend.

---
## Resumen: qué se arregló

| # | Defecto del modelo anterior | Corrección | Dónde |
|---|---|---|---|
| 1 | `dice(máscara, silueta_ropa_vieja)` premiaba copiar la forma anterior | La zona sale de **pose + metadatos de la prenda destino**; el parecido con la ropa vieja ahora se **castiga** | celdas 5 y 11 |
| 2 | La agnóstica solo borraba la ropa vieja dilatada | Zona = superconjunto geométrico; la ropa vieja solo entra como garantía de borrado | celda 5 |
| 3 | `tanh(flujo)*0.22` | `tanh(flujo)*0.60` | celda 10 |
| 4 | Objetivo circular, sin verdad de terreno | **612 pares destilados** de tu catálogo con el modo foto | celda 8 |
| 5 | El cuello era zona preservada ⇒ escote imposible | `neck_type` decide cuánto se libera hacia la barbilla, + `castigo_escote` ×6 | celdas 5 y 11 |
| 6 | `last.pth` de 0 bytes | `guardar()` verifica el tamaño antes de reemplazar | celda 12 |
| 7 | ONNX exportado sin pesos | Tres `assert`: tamaño, equivalencia numérica y fps | celda 13 |

### Orden de ejecución

Celdas 1→6 (~10 min) · 7 (~10 min de descarga) · **8 (~1 h 45 min, lo caro)** · 9→12 (~40 min) ·
13→14 (~5 min).

Si se corta a mitad de la celda 8, vuelve a correrla: salta lo ya destilado.

### Lo que este cuaderno **no** hace

- No reentrena CatVTON. Se usa tal cual; lo que se entrena con tu catálogo es el modelo del
  modo cámara. Si en la defensa te preguntan "¿qué entrenaste tú?", la respuesta honesta es:
  el modelo ligero de 1,8 M de parámetros del modo cámara, destilado del modo foto sobre el
  catálogo FashionStore.
- No toca `resultados_vton/`. Todo lo nuevo va a `resultados_vton_v3/`, así conservas el
  material del modelo anterior para comparar en el informe.